In [24]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

In [88]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [89]:
rng = np.random.default_rng(73512)

In [90]:
# load data from pickle file
raw_data = pd.read_pickle('train.pkl')
np.random.shuffle(raw_data)

In [91]:
classes = [int(raw_data[i][1]) for i in range(len(raw_data))]
data = [torch.tensor(raw_data[i][0]).long() for i in range(len(raw_data))]
targets = torch.tensor(classes, dtype=torch.long)
print(pd.Series(classes).unique())
print(len(data))
print(data[0].shape)
print(data[1])


[0 3 4 1 2]
2939
torch.Size([148])
tensor([112,  48, 160, 112,  48,  37,  37,   5,   5, 168,   9,  50,  37,  37,
        146, 100, 100,  68, 162, 160, 148, 116, 100, 121,   4,  15,   9,   9,
        148, 160,   4,  41,  80,  10, 148, 160, 121,  34,  50, 162, 160,  33,
          4,  33,  50, 160, 148, 114, 180, 180, 180, 162,  56, 160,  52,  90,
         60,  15, 158,  45,  15,  56,  15,  15,  93,  89,   4,   2,   2,   4,
          4,   1,   1,   4,  89,  89,  89,  33, 180,  60,   6,  15, 148,  10,
         82, 148,  10,  82,  10, 148,  50,   7,  42,   4,  48,  48,  34, 100,
        145,  33,  33,  33,  36,  68, 113,   1, 180,   1,   4,  89, 154, 116,
        160,  56,  80,   4,   4,  82, 160,  56,  80,   4,   4,  82, 160, 160,
        160, 160, 160,  82,  82,  82,  82, 160,  56,  33,  33,   4,  33,  33,
          4,  33,   4,  15,  15, 180, 180,  52, 148,  41,   1, 114, 180, 162,
         89, 154,   9, 160,  50,  50, 160,  50,  18,  96,  50,  84,  96,  49,
        162,   3,  82,   3,  

In [92]:
from torch.utils.data import Dataset

class VariableLenDataset(Dataset):
    def __init__(self, in_data, target, transform=None):
        self.data = [(x, y) for x, y in zip(in_data, target)]
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        in_data, target = self.data[idx]
        if self.transform:
            in_data = self.transform(in_data)
        return in_data, target

In [93]:
train_indices = int(len(data) * 0.7)
train_set = VariableLenDataset(data[:train_indices], targets[:train_indices])
valid_set = VariableLenDataset(data[train_indices:], targets[train_indices:])

print(len(train_set[0][0]))
print(len(train_set[1][0]))
train_set[0]

148
747


(tensor([180,  12,  92, 121,  44,   8, 100, 156,  12,  92,   6, 156,  12,  12,
         156, 180, 144,  78,   5, 148,   8, 132,  12, 159,  68,  76,  76, 183,
          76,  92,   5,  44, 151,  92,  44,  28,  92,  44, 120, 125,  33, 172,
          12,  13, 142,  12,  92,  64,  71,  71,   5, 145, 190,  79,  13, 159,
          31,  76,  76,  92,  92,  64, 146,  12,  93,  47, 159,  92,  12,  47,
          20,  12,  77, 141, 185,  79,  12,   5,  92, 132,  92, 110, 111,  74,
          12, 124,  73, 159,  92, 159, 124,  12,  12,  47,  20,  12,  92,  12,
         185,  60,  44,  25,  44, 125,   5,  47,  28,  47,  12,  12,  12,  12,
          13,  13,  28,  12,  12,  92, 124,  73, 159,  92,  12,  47,  20,  12,
         111,  47,  25,  40,  44,  25,  47,  47,  71,  92,  15,  39, 124,  28,
           5,  45,  79,   5,  12,  78,  78, 156]),
 tensor(0))

In [95]:
num_classes = 5

class_counts = torch.bincount(targets[:train_indices], minlength=num_classes)

class_weights = 1.0 / class_counts.float()

# normalizacja opcjonalna
class_weights = class_weights / class_weights.sum() * num_classes

print(class_weights)

tensor([0.1926, 0.6738, 2.1390, 0.7062, 1.2884])


In [96]:
max_value = max([torch.max(d[0]).item() for d in train_set])
min_value = min([torch.min(d[0]).item() for d in train_set])
print(f'Max value: {max_value}, Min value: {min_value}')
VOCAB_SIZE = int(max_value - min_value + 2) # adding also unknown token
print(f'Vocabulary size: {VOCAB_SIZE}')

Max value: 191, Min value: -1
Vocabulary size: 194


In [97]:
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

pad = 0

def pad_collate(batch, pad_value=0):
    xx, yy = zip(*batch)
    x_lens = [len(x) for x in xx]

    xx_pad = pad_sequence(xx, batch_first=True, padding_value=pad_value)

    yy = torch.tensor(yy, dtype=torch.long)

    return xx_pad, yy, x_lens

In [98]:
BATCH_SIZE = 32
HIDDEN_SIZE = 4
EMBEDDING_SIZE = 4
OUT_SIZE = 5
NUM_LAYERS = 2
BIDIRECTIONAL = False
DROPOUT = 0.2
LR = 0.001
TRAIN_EPOCHS = 3

In [99]:
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, collate_fn=pad_collate)
valid_loader = DataLoader(valid_set, batch_size=BATCH_SIZE, shuffle=False, drop_last=False, collate_fn=pad_collate)

In [ ]:
batch_x, batch_y, lens = next(iter(train_loader))
next(iter(train_loader))

(tensor([[ -1,  -1, 112,  ...,   0,   0,   0],
         [112,   0,   1,  ...,  -1, 144, 144],
         [  1, 146,  36,  ...,   0,   0,   0],
         ...,
         [ 50,   5,  50,  ...,   0,   0,   0],
         [145,  12,  12,  ...,   0,   0,   0],
         [ -1,  -1,  -1,  ...,   0,   0,   0]]),
 tensor([0, 1, 0, 0, 4, 1, 4, 3, 0, 0, 0, 3, 0, 1, 3, 3, 0, 2, 0, 3, 1, 1, 0, 0,
         1, 0, 3, 1, 4, 0, 0, 0]),
 [312,
  2726,
  688,
  36,
  461,
  160,
  276,
  615,
  76,
  336,
  252,
  237,
  138,
  2556,
  208,
  408,
  228,
  182,
  204,
  384,
  1260,
  687,
  159,
  1215,
  711,
  83,
  176,
  1229,
  304,
  260,
  208,
  580])

In [108]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, emb_size, hidden_size, num_layers, out_size, min_vocab_value=-1, max_vocab_value=191, bidirectional = False):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_size = hidden_size
        self.min_vocab_value = min_vocab_value
        self.max_vocab_value = max_vocab_value
        self.unk_idx = max_vocab_value - min_vocab_value + 1
        if bidirectional:
            self.bidirectional = 2
        else:
            self.bidirectional = 1
        self.embedding = nn.Embedding(vocab_size, emb_size)
        self.lstm = nn.LSTM(input_size = emb_size, hidden_size = hidden_size, num_layers = num_layers, bidirectional=bidirectional, dropout=DROPOUT)
        self.fc = nn.Linear(hidden_size*self.bidirectional, out_size) # do poprawy jeśli chce się korzystać z czego innego
        
    def init_hidden(self, batch_size):
        hidden = torch.zeros(self.num_layers*self.bidirectional , batch_size, self.hidden_size)
        state = torch.zeros(self.num_layers*self.bidirectional , batch_size, self.hidden_size)
        return hidden, state
    
    def forward(self, x, x_len, hidden):

        invalid = (x < self.min_vocab_value) | (x > self.max_vocab_value)
        x = x.clone()
        x[invalid] = self.unk_idx + self.min_vocab_value
        print('-- DEBUG --')
        print("UNK ratio:", (x == self.unk_idx + self.min_vocab_value).float().mean())
        print("invalid ratio:", invalid.float().mean())
        print(min(x_len), max(x_len))
        print('' if x.shape[1] == max(x_len) else 'strange')
        x = self.embedding(x - self.min_vocab_value)
        x =x.squeeze(2)
        packed = nn.utils.rnn.pack_padded_sequence(x, x_len, batch_first=True, enforce_sorted=False)
        packed_out, (hn, cn) = self.lstm(packed, hidden)
        # print(f'{hn.shape=}')
        if self.bidirectional == 1:
            out = hn[-1]
        else:
            forward_last = hn[-2]
            backward_last = hn[-1]
            out = torch.cat((forward_last, backward_last), dim=1)
                            
        # print(f'{out.shape=}')
        return self.fc(out), (hn, cn)
    
model = LSTMClassifier(VOCAB_SIZE, EMBEDDING_SIZE, HIDDEN_SIZE, NUM_LAYERS, OUT_SIZE, min_vocab_value=min_value, max_vocab_value=max_value, bidirectional=BIDIRECTIONAL).to(device)
model

LSTMClassifier(
  (embedding): Embedding(194, 4)
  (lstm): LSTM(4, 4, num_layers=2, dropout=0.2)
  (fc): Linear(in_features=4, out_features=5, bias=True)
)

In [109]:
optimizer = torch.optim.Adam(model.parameters(), lr = LR)
loss_fun = nn.CrossEntropyLoss(weight=class_weights)

from tqdm import tqdm
# Training loop
model.train()
for epoch in tqdm(range(TRAIN_EPOCHS)):
    # for x, targets, x_len in train_loader:
    x, targets, x_len = batch_x, batch_y, lens
    x = x.to(device).unsqueeze(2)
    targets = targets.to(device)
    hidden, state = model.init_hidden(x.size(0))
    hidden, state = hidden.to(device), state.to(device) 
    preds, _ = model(x, x_len, (hidden,state))
    loss = loss_fun(preds, targets)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad() 
    if epoch % 1 == 0:
        print(f"Epoch: {epoch}, loss: {loss.item():.3}")

  0%|          | 0/3 [00:00<?, ?it/s]

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
68 2313

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
44 2008

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
44 4082

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
64 1372

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
32 2112

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
52 1576

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
56 1430

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
56 2044

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
36 1701

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
56 1416

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
48 1479

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
40 860

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
28 1916

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
64 1542

-- DEBUG --
UNK ratio: tensor(0.)
i

 33%|███▎      | 1/3 [01:44<03:28, 104.11s/it]

Epoch: 0, loss: 1.65
-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
36 2502

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
48 3626

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
64 3313

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
36 2315

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
52 2661

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
6 1422

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
64 2589

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
52 4082

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
44 4016

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
44 1542

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
40 4969

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
44 2172

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
48 2345

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
13 2219

-- DEBUG --
UN

 67%|██████▋   | 2/3 [03:45<01:53, 113.99s/it]

Epoch: 1, loss: 1.64
-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
56 4596

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
16 1422

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
56 2878

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
32 2502

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
68 4082

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
36 1598

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
56 6308

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
12 1215

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
44 2320

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
20 2197

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
36 1302

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
56 877

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
12 2112

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
16 2120

-- DEBUG --
UN

100%|██████████| 3/3 [05:58<00:00, 119.64s/it]

Epoch: 2, loss: 1.62


In [110]:
model.eval()
preds= []
targets= []

with torch.no_grad():
    for x, target, x_len in valid_loader:
        x = x.to(device).unsqueeze(2)
        target = target.to(device)
        hidden, state = model.init_hidden(x.shape[0])
        hidden, state = hidden.to(device), state.to(device)
        pred, _ = model(x, x_len, (hidden, state))
        preds.append(pred.cpu())
        targets.append(target.cpu())
print(f"Accuracy: {(torch.argmax(torch.cat(preds),1).cpu()==torch.cat(targets)).sum().item()/len(torch.cat(targets)):.3}")

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
32 1932



-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
48 3489

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
12 1458

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
40 2124

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
16 1629

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
44 1340

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
52 2084

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
76 2292

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
40 2999

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
44 1086

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
28 2975

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
20 2268

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
6 2980

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
16 4329

-- DEBUG --
UNK ratio: tensor(0.)
invalid ratio: tensor(0.)
18 3323

-- DEBUG --
UNK ratio: tensor(0.)
i

In [111]:
print(torch.argmax(torch.cat(preds),1).cpu())

tensor([0, 1, 1, 0, 4, 0, 0, 1, 0, 0, 0, 1, 1, 0, 4, 0, 3, 0, 0, 0, 0, 4, 0, 1,
        0, 0, 4, 0, 0, 0, 1, 3, 0, 0, 3, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0,
        0, 1, 4, 1, 0, 1, 3, 0, 4, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1,
        0, 0, 0, 4, 4, 0, 0, 0, 4, 4, 0, 0, 1, 3, 0, 0, 0, 1, 0, 0, 0, 0, 0, 4,
        0, 3, 0, 0, 3, 0, 3, 0, 0, 4, 3, 0, 4, 0, 0, 4, 0, 1, 0, 1, 3, 4, 0, 0,
        1, 0, 0, 4, 3, 1, 0, 0, 3, 3, 0, 1, 0, 1, 4, 0, 0, 0, 0, 1, 0, 0, 1, 1,
        4, 0, 0, 3, 1, 0, 1, 0, 4, 1, 1, 3, 1, 0, 0, 0, 0, 4, 0, 4, 0, 1, 0, 0,
        0, 0, 3, 0, 1, 1, 0, 1, 0, 4, 1, 0, 0, 0, 3, 1, 0, 0, 4, 0, 4, 4, 0, 1,
        1, 0, 0, 0, 0, 0, 4, 3, 0, 1, 0, 0, 0, 1, 0, 3, 0, 0, 4, 1, 0, 0, 0, 0,
        0, 0, 0, 3, 3, 0, 0, 4, 1, 4, 0, 1, 4, 0, 3, 0, 1, 1, 0, 1, 1, 0, 1, 3,
        1, 4, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 3, 0, 0, 0, 3, 4, 1, 0, 0, 0, 4,
        0, 0, 0, 4, 0, 0, 0, 4, 0, 1, 0, 0, 3, 0, 0, 0, 1, 0, 4, 1, 3, 3, 0, 0,
        0, 4, 4, 4, 0, 4, 1, 0, 0, 0, 0,

In [112]:
print((torch.argmax(torch.cat(preds),1).cpu() == 0).sum())
print((torch.argmax(torch.cat(preds),1).cpu() != 0).sum())

tensor(489)
tensor(393)
